# TAE-IA · Module 6 · L08 — Neural Style Transfer

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L08 |
| **Track** | A — Vision |
| **Estimated duration** | 2 hours |
| **GPU required** | T4 (Colab) |
| **Prerequisites** | L01–L07 completed; L07 cleanup cell run |

## Learning objectives
By the end of this notebook you will be able to:
- [ ] Implement classic NST using VGG19 content and style losses
- [ ] Construct a Gram matrix and explain what it captures about an image
- [ ] Apply fast style transfer via `torch.hub` and compare speed and quality
- [ ] Position each approach (classic NST, fast NST, SD img2img) for a real deployment scenario

## Before you start
- T4 GPU runtime selected
- L07 cleanup cell was run (frees ~130 MB)

---

## Cell 0 — Setup (always run this first)

> Mounts Drive, checks GPU, fixes seed, and logs in to HuggingFace.

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random, time, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'   # persistent, survives sessions
LOCAL_CACHE = '/content/model_cache'                      # ephemeral, dies with the runtime
os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Clear any cache left by earlier course versions (the old HF-hub cache location only --
# torch.hub's cache below lives in its own subfolder and is untouched by this).
for _leftover in ('hub', 'xet'):
    _p = os.path.join(MODEL_CACHE, _leftover)
    if os.path.exists(_p):
        shutil.rmtree(_p)

# ----------------------------------------------------------------
# Model cache with automatic fallback
# ----------------------------------------------------------------
# Models are cached on Drive so they survive a session restart. A free
# Google account has only 15 GB, shared with Gmail and Photos, which is
# less than this course needs end to end. So when Drive has no room we
# cache on the runtime's own disk instead: the lab still runs, but that
# copy is deleted when the session ends.
#
# The choice is made per model, not once per notebook. A model already
# sitting on Drive keeps being read from Drive even when Drive is far
# too full to accept anything new.

DRIVE_HEADROOM_GB = 1.0    # never fill Drive to the last byte
SIZE_MARGIN       = 1.15   # temp files and metadata written during a download


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 0.0


def _is_cached(path):
    """True if <path> holds anything but HuggingFace's incomplete-download folder."""
    return os.path.isdir(path) and any(e != '.cache' for e in os.listdir(path))


def _disk_full(exc):
    """ENOSPC, quota exceeded, or the I/O error drivefs raises when Drive is full."""
    return (getattr(exc, 'errno', None) in (28, 122, 5)
            or 'no space' in str(exc).lower()
            or 'quota' in str(exc).lower())


def cache_dir(name, needed_gb):
    """Return the directory model <name> should live in, preferring Drive."""
    drive_dir = os.path.join(MODEL_CACHE, name)
    local_dir = os.path.join(LOCAL_CACHE, name)

    if _is_cached(drive_dir):
        return drive_dir            # already on Drive: reading it costs no space
    if _is_cached(local_dir):
        return local_dir            # already fell back earlier this session

    required = needed_gb * SIZE_MARGIN + DRIVE_HEADROOM_GB
    free = _free_gb(MODEL_CACHE)
    if free >= required:
        os.makedirs(drive_dir, exist_ok=True)
        print(f'[cache] {name} -> Drive ({needed_gb:.2f} GB needed, {free:.1f} GB free).')
        return drive_dir

    local_free = _free_gb('/content')
    if local_free < required:
        raise RuntimeError(
            f'{name} needs ~{needed_gb:.1f} GB, but only {free:.1f} GB is free on '
            f'Drive and {local_free:.1f} GB on the runtime disk.\n'
            f'Free space in Google Drive (delete unused TAE_IA_M6/models subfolders) '
            f'and re-run this cell.')

    os.makedirs(local_dir, exist_ok=True)
    print(f'[cache] Not enough room on Drive for {name}: needs {needed_gb:.2f} GB\n'
          f'        plus margin, {free:.1f} GB free. Caching on the runtime instead.\n'
          f'        The lab runs normally, but this copy is deleted when the session\n'
          f'        ends and downloads again next time. Free space in Drive to avoid\n'
          f'        the repeat download.')
    return local_dir


def cached_fetch(name, needed_gb, download):
    """Run download(target_dir) in the best available cache and return its result.

    Retries on the runtime disk if Drive fills up mid-download: a pre-flight
    space check cannot catch a quota that runs out halfway through.
    """
    target = cache_dir(name, needed_gb)
    try:
        return download(target)
    except OSError as e:
        if not _disk_full(e) or target.startswith(LOCAL_CACHE):
            raise
        print(f'[cache] Drive ran out of room mid-download ({e}).\n'
              f'        Discarding the partial copy and retrying on the runtime disk.')
        shutil.rmtree(target, ignore_errors=True)
        fallback = os.path.join(LOCAL_CACHE, name)
        os.makedirs(fallback, exist_ok=True)
        return download(fallback)


def cached_snapshot(name, repo_id, needed_gb, **kwargs):
    """snapshot_download into the best available cache, returning its path."""
    from huggingface_hub import snapshot_download

    def _dl(target):
        snapshot_download(repo_id, local_dir=target, **kwargs)
        return target

    return cached_fetch(name, needed_gb, _dl)


_drive_free = _free_gb(MODEL_CACHE)
print(f'Model cache: {MODEL_CACHE}  ({_drive_free:.1f} GB free on Drive)')
if _drive_free < 1.0:
    print('WARNING: under 1 GB free on Drive. Models will cache on the runtime,\n'
          '         but saving your lab outputs may fail. Free space in Drive.')

# torch.hub / torchvision downloads (VGG19, fast-NST) are plain files, no symlinks --
# unlike HF_HOME, TORCH_HOME is safe to point at Drive and is kept here on purpose.
os.environ['TORCH_HOME'] = cache_dir('torch_home', 0.55)

import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Go to: Runtime > Change runtime type > T4 GPU')
    raise SystemExit('GPU required.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f'Fixed seed: {SEED}')
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')

# HuggingFace login -- needed every new Colab session
import huggingface_hub

try:
    _token = huggingface_hub.get_token()
except Exception:
    _token = None

if _token:
    print(f"Already logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
else:
    try:
        from google.colab import userdata
        _hf_token = userdata.get('HF_TOKEN')
    except Exception:
        _hf_token = None

    if _hf_token:
        huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
        print(f"Logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
    else:
        raise RuntimeError(
            'No HF_TOKEN found in Colab Secrets (key icon, left sidebar).\n'
            'Add a secret named HF_TOKEN with your HuggingFace read token, enable notebook access, '
            'then re-run this cell.\n'
            'See L00 Cell 4 if you need to generate a token or accept the SD 1.5 license.'
        )

In [ ]:
# ================================================================
# Install dependencies for L08
# (torchvision is pre-installed on Colab — no pip needed for classic NST)
# diffusers needed for the img2img comparison in Section 2.4
# ================================================================
!pip install diffusers transformers accelerate -q

import torchvision
import diffusers
print(f'torchvision {torchvision.__version__}  |  diffusers {diffusers.__version__}')

---
## Part 1 — Context and Key Concepts

> Read this before running any code.

### What VGG features represent

VGG19 is a classification network trained on ImageNet. Its internal activations at different layers encode image information at different abstraction levels:

- **Early layers (conv1, conv2):** detect local edges, colors, and small textures. High spatial resolution — they know *where* textures appear.
- **Deep layers (conv4, conv5):** detect semantic objects and compositional structure. Low spatial resolution — they know *what* is in the image.

Style transfer re-uses these frozen weights. No training happens during inference — only the pixel values of the output image are updated.

### The Gram matrix: why it captures style

The Gram matrix $G^l$ at layer $l$ is the inner product of the feature map with itself across spatial positions:

$$G^l_{ij} = \frac{1}{CHW} \sum_k F^l_{ik} \cdot F^l_{jk}$$

Entry $G_{ij}$ measures how strongly channels $i$ and $j$ activate together, averaged over all spatial locations. By averaging over space, the Gram matrix **discards layout information** while keeping texture co-occurrence. A painting's brushstroke texture appears everywhere, not in one spot — the Gram matrix captures this global quality.

### Total loss and the α/β ratio

$$\mathcal{L}_{total} = \alpha \cdot \mathcal{L}_{content} + \beta \cdot \mathcal{L}_{style}$$

- **Low β/α (e.g., 1e4):** output looks like the content image with a slight texture tint.
- **High β/α (e.g., 1e8):** content structure is destroyed; output is pure style texture.
- **Typical sweet spot:** β/α ≈ 1e6–1e7 for most content/style pairs.

> These numbers are on this notebook's scale: `gram_matrix` divides the Gram by `C·H·W`, which shrinks the style loss by two to three orders of magnitude. Papers and tutorials that use raw Gram matrices quote β/α ≈ `1e3–1e4` for the same visual balance — same ratio of forces, different units.

---

## Part 2 — Lab

### Section 2.0 — Load VGG19 and image utilities

In [ ]:
# Section 2.0 — Load VGG19 and image utilities
import torch
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import requests

DEVICE = 'cuda'
IMG_SIZE = 512
OUTPUT_DIR = '/content/drive/MyDrive/TAE_IA_M6/L08_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# VGG19 — features only (no classifier)
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(DEVICE).eval()
for param in vgg.parameters():
    param.requires_grad_(False)
print(f'VGG19 loaded. Feature layers: {len(list(vgg.children()))}')

# Image transforms
mean = torch.tensor([0.485, 0.456, 0.406]).to(DEVICE)
std  = torch.tensor([0.229, 0.224, 0.225]).to(DEVICE)

to_tensor = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

# Wikimedia rejects the default requests User-Agent with HTTP 403, so any
# fetch aimed at it has to identify itself.
WIKI_UA = {'User-Agent': 'TAE-IA-M6-course/1.0 (classroom use; contact instructor)'}

def load_image(url_or_path):
    if url_or_path.startswith('http'):
        resp = requests.get(url_or_path, timeout=20, headers=WIKI_UA)
        resp.raise_for_status()
        img  = Image.open(BytesIO(resp.content)).convert('RGB')
    else:
        img  = Image.open(url_or_path).convert('RGB')
    return img

def img_to_tensor(pil_img):
    t = to_tensor(pil_img).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)
    return (t - mean[None, :, None, None]) / std[None, :, None, None]

def tensor_to_img(t):
    # mean/std live on the GPU (they normalise on the GPU), so they have to
    # follow the tensor to the CPU here -- mixing a cuda tensor with a cpu one
    # raises "Expected all tensors to be on the same device".
    t = t.squeeze(0).detach().cpu()
    m, s = mean.cpu(), std.cpu()
    t = t * s[:, None, None] + m[:, None, None]
    t = t.clamp(0, 1)
    return T.ToPILImage()(t)

print('Utilities ready.')

### Section 2.1 — Load the content and style images

In [ ]:
# Section 2.1 — Load the content and style images
# Students: replace URLs with any images you prefer.
# Content: a photo with clear structure (landscape, portrait, architecture)
# Style:   an artwork with distinctive texture (impressionist, geometric, abstract)

# Wikimedia serves only a fixed set of thumbnail widths
# (20/40/60/120/250/330/500/960/1280/1920/3840); any other width returns HTTP 400.
CONTENT_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/f/f4/Honeycrisp.jpg/500px-Honeycrisp.jpg'
STYLE_URL   = 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/960px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg'

# Already have images on your computer? Put their paths here and the download
# is skipped entirely (load_image() accepts local paths as well as URLs).
CONTENT_PATH = None    # e.g. '/content/my_photo.jpg'
STYLE_PATH   = None    # e.g. '/content/my_painting.jpg'


def upload_image(label='image'):
    """Ask the browser for a file and return it as a PIL image (Colab only)."""
    from google.colab import files as colab_files
    print(f'>>> Choose a {label} image (JPG/PNG) from your computer...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError(
            f'No {label} image was chosen. Re-run this cell and pick a file, or set '
            f'{label.upper()}_PATH at the top of this cell to an image you uploaded '
            'through the Colab file browser (folder icon, left sidebar).')
    name = next(iter(uploaded))
    img  = Image.open(BytesIO(uploaded[name])).convert('RGB')
    print(f'{label}: {name}  {img.size}')
    return img


def safe_load(url, label='image', local_path=None, attempts=3):
    """Fetch an image; if the download keeps failing, ask you to upload one.

    It never falls back to a random-noise image. That is worse than a crash:
    style transfer runs happily on noise and produces a garbage result with no
    error anywhere, so you cannot tell a broken download from a bad style
    weight. An upload is fine because you chose the image yourself.
    """
    if local_path:
        img = load_image(local_path)
        print(f'{label}: {local_path}  {img.size}')
        return img

    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            img = load_image(url)
            print(f'{label}: {img.size}')
            return img
        except Exception as e:
            last_error = e
            status = getattr(getattr(e, 'response', None), 'status_code', None)
            print(f'{label}: download failed (attempt {attempt}/{attempts}) — {e}')
            # 429 = Wikimedia rate limit. Every Colab notebook in the room goes
            # out through a handful of shared IPs, so a whole class trips the cap
            # at once and waiting a few seconds is usually enough. A 404 or a bad
            # URL will never fix itself, so do not retry those.
            if attempt < attempts and status in (429, 500, 502, 503, 504):
                wait = 5 * attempt
                print(f'   rate-limited by the server — retrying in {wait}s...')
                time.sleep(wait)
            else:
                break

    print(f'\nGiving up on the {label} URL:\n  {url}')
    try:
        return upload_image(label)
    except ImportError:
        raise RuntimeError(
            f'Could not download the {label} image ({last_error}), and the Colab '
            f'upload widget is not available outside Colab.\n'
            f'Fix: set {label.upper()}_PATH at the top of this cell to a local image file.'
        ) from last_error

content_pil = safe_load(CONTENT_URL, label='content', local_path=CONTENT_PATH)
style_pil   = safe_load(STYLE_URL,   label='style',   local_path=STYLE_PATH)

content_tensor = img_to_tensor(content_pil)
style_tensor   = img_to_tensor(style_pil)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(content_pil.resize((512, 512))); axes[0].set_title('Content image'); axes[0].axis('off')
axes[1].imshow(style_pil.resize((512, 512)));   axes[1].set_title('Style image');   axes[1].axis('off')
plt.tight_layout(); plt.show()

### Section 2.2 — Classic style transfer: VGG19 optimization loop

In [ ]:
# Section 2.2 — Feature extractor and loss functions

# Map VGG19 layer indices to readable names
# (index 0 = first conv layer of vgg.features)
CONTENT_LAYERS = {'21': 'conv4_2'}                          # 1 deep layer
STYLE_LAYERS   = {'0':  'conv1_1', '5':  'conv2_1',         # 5 early layers
                  '10': 'conv3_1', '19': 'conv4_1',
                  '28': 'conv5_1'}
ALL_LAYERS     = set(CONTENT_LAYERS) | set(STYLE_LAYERS)

def extract_features(img_tensor, model):
    """Run forward pass and return dict of named layer outputs."""
    features = {}
    x = img_tensor
    for idx, layer in enumerate(model):
        x = layer(x)
        key = str(idx)
        if key in ALL_LAYERS:
            features[key] = x
    return features

def gram_matrix(features):
    # Dividing by C*H*W keeps the loss comparable across layers of very
    # different sizes, but it also shrinks the style loss by several orders of
    # magnitude relative to the raw Gram matrix in Gatys et al. That is why the
    # style weights in this notebook look so much larger than the beta/alpha
    # ratios of 1e3-1e4 quoted in the paper: same balance, different scale.
    B, C, H, W = features.shape
    f = features.view(B, C, H * W)
    G = torch.bmm(f, f.transpose(1, 2))
    return G / (C * H * W)

def compute_style_loss(gen_feats, style_grams):
    loss = 0.0
    for layer in STYLE_LAYERS:
        G_gen   = gram_matrix(gen_feats[layer])
        G_style = style_grams[layer]
        loss   += torch.mean((G_gen - G_style) ** 2)
    return loss / len(STYLE_LAYERS)

def compute_content_loss(gen_feats, content_feats):
    layer = list(CONTENT_LAYERS.keys())[0]   # conv4_2
    return torch.mean((gen_feats[layer] - content_feats[layer]) ** 2)

# Pre-compute target features (run once — these don't change)
with torch.no_grad():
    content_feats = extract_features(content_tensor, vgg)
    style_feats   = extract_features(style_tensor,   vgg)
    style_grams   = {k: gram_matrix(v).detach() for k, v in style_feats.items()}

print('Feature extractor and loss functions ready.')

In [ ]:
# Section 2.2 (cont.) — Style transfer optimization loop

def run_style_transfer(content_tensor, style_grams, content_feats,
                       content_weight=1.0, style_weight=1e6,
                       num_steps=300, show_every=100):
    # Initialize generated image as a copy of the content image
    generated = content_tensor.clone().requires_grad_(True)

    # LBFGS: well-suited for this constrained optimization
    optimizer = torch.optim.LBFGS([generated], lr=1.0)

    step      = [0]
    snapshots = []

    def closure():
        optimizer.zero_grad()
        gen_feats = extract_features(generated, vgg)
        c_loss    = compute_content_loss(gen_feats, content_feats)
        s_loss    = compute_style_loss(gen_feats, style_grams)
        loss      = content_weight * c_loss + style_weight * s_loss
        loss.backward()

        step[0] += 1
        if step[0] % show_every == 0 or step[0] == 1:
            print(f"  Step {step[0]:4d} | total={loss.item():.2f} "
                  f"| content={c_loss.item():.4f} | style={s_loss.item():.6f}")
            snapshots.append((step[0], tensor_to_img(generated)))
        return loss

    t0 = time.time()
    for _ in range(num_steps):
        optimizer.step(closure)
        if step[0] >= num_steps:
            break
    t = time.time() - t0
    print(f"Done. {num_steps} steps in {t:.1f}s")

    return tensor_to_img(generated), snapshots

print("Running classic NST — 300 steps (~30-60 s on T4)...")
result_classic, snapshots = run_style_transfer(
    content_tensor, style_grams, content_feats,
    content_weight=1.0, style_weight=1e6, num_steps=300
)
result_classic.save(os.path.join(OUTPUT_DIR, 'classic_nst_result.png'))

# Show convergence snapshots
n = len(snapshots)
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
axes[0].imshow(content_pil.resize((512, 512))); axes[0].set_title('Content'); axes[0].axis('off')
for ax, (step_n, img) in zip(axes[1:], snapshots):
    ax.imshow(img); ax.set_title(f'Step {step_n}'); ax.axis('off')
plt.suptitle('Classic NST convergence (content_w=1, style_w=1e6)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'classic_nst_convergence.png'), dpi=80)
plt.show()

**What do you observe?**  
- At what step does the style become clearly visible?
- What changes first — color palette, brushstroke texture, or overall composition?
- Is there a step where the content structure starts to degrade?

*Write your observation here:*

(double-click to edit)

### Section 2.3 — Style weight sweep: finding the α/β sweet spot

In [ ]:
# Section 2.3 — Style weight sweep
# Runs 4 variations: this takes ~2 minutes total on T4
style_weights = [1e5, 1e6, 1e7, 1e8]
weight_results = {}

for sw in style_weights:
    print(f"\nStyle weight = {sw:.0e}")
    img, _ = run_style_transfer(
        content_tensor, style_grams, content_feats,
        content_weight=1.0, style_weight=sw, num_steps=200, show_every=200
    )
    weight_results[sw] = img
    img.save(os.path.join(OUTPUT_DIR, f'nst_style_w{sw:.0e}.png'))

fig, axes = plt.subplots(1, len(style_weights) + 1, figsize=(4 * (len(style_weights) + 1), 4))
axes[0].imshow(content_pil.resize((512, 512))); axes[0].set_title('Content'); axes[0].axis('off')
for ax, (sw, img) in zip(axes[1:], weight_results.items()):
    ax.imshow(img); ax.set_title(f'β = {sw:.0e}'); ax.axis('off')
plt.suptitle('Style weight sweep (α=1, num_steps=200)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'style_weight_sweep.png'), dpi=80)
plt.show()

**What do you observe?**  
- At what β value does the content become unrecognizable?
- Which β value gives the best visual balance for your specific content/style pair?
- Does the optimal β depend on the complexity of the content image or the style image?

*Write your observation here:*

(double-click to edit)

### Section 2.4 — Fast style transfer with a feedforward network

In [ ]:
# Section 2.4 — Fast NST (Johnson et al. 2016 feedforward network)
#
# This used to call torch.hub.load('pytorch/examples:main', 'fast_neural_style').
# That entry point does not exist: the pytorch/examples repository has no
# hubconf.py, so every style raised and the whole section silently produced
# nothing. We instead define the network here and load the authors' released
# weights directly.
import re, zipfile, urllib.request
import torch.nn as nn

# ---- TransformerNet: the architecture the released checkpoints were trained with
#      (verbatim from pytorch/examples/fast_neural_style/neural_style/transformer_net.py)
class ConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super().__init__()
        self.reflection_pad = nn.ReflectionPad2d(kernel_size // 2)
        self.conv2d = nn.Conv2d(in_channels, out_channels, kernel_size, stride)

    def forward(self, x):
        return self.conv2d(self.reflection_pad(x))


class UpsampleConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, upsample=None):
        super().__init__()
        self.upsample = upsample
        self.reflection_pad = nn.ReflectionPad2d(kernel_size // 2)
        self.conv2d = nn.Conv2d(in_channels, out_channels, kernel_size, stride)

    def forward(self, x):
        if self.upsample:
            x = nn.functional.interpolate(x, mode='nearest', scale_factor=self.upsample)
        return self.conv2d(self.reflection_pad(x))


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in1   = nn.InstanceNorm2d(channels, affine=True)
        self.conv2 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in2   = nn.InstanceNorm2d(channels, affine=True)
        self.relu  = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.in1(self.conv1(x)))
        return self.in2(self.conv2(out)) + x


class TransformerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = ConvLayer(3, 32, kernel_size=9, stride=1)
        self.in1   = nn.InstanceNorm2d(32, affine=True)
        self.conv2 = ConvLayer(32, 64, kernel_size=3, stride=2)
        self.in2   = nn.InstanceNorm2d(64, affine=True)
        self.conv3 = ConvLayer(64, 128, kernel_size=3, stride=2)
        self.in3   = nn.InstanceNorm2d(128, affine=True)
        self.res1, self.res2 = ResidualBlock(128), ResidualBlock(128)
        self.res3, self.res4 = ResidualBlock(128), ResidualBlock(128)
        self.res5 = ResidualBlock(128)
        self.deconv1 = UpsampleConvLayer(128, 64, kernel_size=3, stride=1, upsample=2)
        self.in4     = nn.InstanceNorm2d(64, affine=True)
        self.deconv2 = UpsampleConvLayer(64, 32, kernel_size=3, stride=1, upsample=2)
        self.in5     = nn.InstanceNorm2d(32, affine=True)
        self.deconv3 = ConvLayer(32, 3, kernel_size=9, stride=1)
        self.relu    = nn.ReLU()

    def forward(self, X):
        y = self.relu(self.in1(self.conv1(X)))
        y = self.relu(self.in2(self.conv2(y)))
        y = self.relu(self.in3(self.conv3(y)))
        y = self.res5(self.res4(self.res3(self.res2(self.res1(y)))))
        y = self.relu(self.in4(self.deconv1(y)))
        y = self.relu(self.in5(self.deconv2(y)))
        return self.deconv3(y)


# ---- Weights: ~25 MB zip of the four styles, cached on Drive after the first run
FAST_NST_DIR = cache_dir('fast_neural_style', 0.06)
FAST_NST_ZIP = os.path.join(FAST_NST_DIR, 'saved_models.zip')
FAST_NST_URL = 'https://www.dropbox.com/s/lrvwfehqdcxoza8/saved_models.zip?dl=1'

fast_styles = ['mosaic', 'candy', 'rain_princess', 'udnie']

if not all(os.path.exists(os.path.join(FAST_NST_DIR, f'{s}.pth')) for s in fast_styles):
    print('Downloading fast-NST weights (~25 MB, once per Drive)...')
    urllib.request.urlretrieve(FAST_NST_URL, FAST_NST_ZIP)
    with zipfile.ZipFile(FAST_NST_ZIP) as zf:
        for member in zf.namelist():
            if member.endswith('.pth'):
                with zf.open(member) as fsrc, \
                     open(os.path.join(FAST_NST_DIR, os.path.basename(member)), 'wb') as fdst:
                    fdst.write(fsrc.read())
    os.remove(FAST_NST_ZIP)
print(f'Weights ready in {FAST_NST_DIR}')


def load_fast_nst(style_name):
    """Load one pre-trained style network."""
    ckpt = os.path.join(FAST_NST_DIR, f'{style_name}.pth')
    try:
        state = torch.load(ckpt, map_location='cpu', weights_only=True)
    except Exception:
        # These checkpoints predate the zip serialisation format (pickle
        # protocol 2, 2018). If the restricted unpickler cannot read one, fall
        # back -- we downloaded the file ourselves from the official
        # pytorch/examples distribution moments ago.
        state = torch.load(ckpt, map_location='cpu', weights_only=False)
    # The released checkpoints carry deprecated InstanceNorm running stats that
    # today's nn.InstanceNorm2d(affine=True) does not register.
    for k in [k for k in state if re.search(r'in\d+\.running_(mean|var)$', k)]:
        del state[k]
    net = TransformerNet()
    net.load_state_dict(state)
    return net.to(DEVICE).eval()


fast_results = {}

# Fast NST was trained on 0-255 inputs, not ImageNet-normalised ones.
content_for_fast = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
])(content_pil).unsqueeze(0).to(DEVICE) * 255.0

for style_name in fast_styles:
    model_fast = load_fast_nst(style_name)
    with torch.no_grad():
        torch.cuda.synchronize()
        t0     = time.time()
        output = model_fast(content_for_fast)
        torch.cuda.synchronize()
        t_fast = time.time() - t0

    out_pil = T.ToPILImage()(output.squeeze(0).clamp(0, 255).byte().cpu())
    fast_results[style_name] = (out_pil, t_fast)
    out_pil.save(os.path.join(OUTPUT_DIR, f'fast_nst_{style_name}.png'))
    print(f'{style_name}: {t_fast*1000:.1f} ms')

    del model_fast
    torch.cuda.empty_cache()

n = len(fast_results)
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
axes[0].imshow(content_pil.resize((512, 512))); axes[0].set_title('Content'); axes[0].axis('off')
for ax, (name, (img, t)) in zip(axes[1:], fast_results.items()):
    ax.imshow(img); ax.set_title(f'{name}\n{t*1000:.0f} ms'); ax.axis('off')
plt.suptitle('Fast NST — four pre-trained styles (single forward pass)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fast_nst_grid.png'), dpi=80)
plt.show()

### Section 2.5 — Three-way comparison: classic NST vs. fast NST vs. SD img2img

In [ ]:
# Section 2.5 — SD img2img as a stylizer
# Uses the SD 1.5 weights already cached in Drive from L01–L06
import gc
from diffusers import StableDiffusionImg2ImgPipeline

# Free VGG VRAM first
del vgg; gc.collect(); torch.cuda.empty_cache()

# Fetch only the fp16 weights + configs (~2.7 GB) -- shared with L01-L06 if already cached.
SD15_DIR = cached_snapshot(
    'sd15-local',
    "runwayml/stable-diffusion-v1-5",
    needed_gb=2.74,
    allow_patterns=["*.json", "*.txt", "*.fp16.safetensors"],
)

pipe_i2i = StableDiffusionImg2ImgPipeline.from_pretrained(
    SD15_DIR,
    torch_dtype=torch.float16,
    variant="fp16",
).to(DEVICE)

# Style prompt matching the Van Gogh Starry Night style reference
# Name the subject, not only the style. Given a medium and an artist alone,
# SD draws the most likely thing that description fits -- a canvas on an easel,
# a framed painting on a wall -- and the apple vanishes as soon as strength is
# high enough to redraw structure. Change the subject here if you change the
# content image.
STYLE_PROMPT = (
    "a red apple on straw, oil painting in the style of Van Gogh, "
    "swirling expressive brushstrokes, vivid blues and yellows, "
    "post-impressionist, thick impasto texture"
)

content_for_sd = content_pil.resize((512, 512))

strength_results = {}
for strength in [0.5, 0.65, 0.8]:
    t0 = time.time()
    for attempt in range(3):
        out = pipe_i2i(
            prompt         = STYLE_PROMPT,
            image          = content_for_sd,
            strength       = strength,
            guidance_scale = 7.5,
            generator      = torch.Generator(DEVICE).manual_seed(SEED + attempt)
        )
        flagged = bool(out.nsfw_content_detected) and out.nsfw_content_detected[0]
        if not flagged:
            break
        # SD 1.5's safety checker compares the output embedding against a set of
        # unsafe concept embeddings, and close-up organic texture -- fruit, skin,
        # flesh tones -- lands close enough to trip it. When it fires you get a
        # solid black image back. This is the false-positive side of the filter
        # you critiqued in L01, so we keep the checker on and re-roll the seed
        # rather than switching it off.
        print(f'  safety checker flagged strength={strength} at seed '
              f'{SEED + attempt} (black image) - retrying with a new seed')
    else:
        print(f'  strength={strength} was flagged on all 3 seeds; '
              f'its panel below stays black.')
    img = out.images[0]
    t = time.time() - t0
    strength_results[strength] = (img, t)
    img.save(os.path.join(OUTPUT_DIR, f'img2img_strength{int(strength*100)}.png'))
    print(f'SD img2img strength={strength}: {t:.1f}s')

fig, axes = plt.subplots(1, len(strength_results) + 1, figsize=(4 * (len(strength_results) + 1), 4))
axes[0].imshow(content_for_sd); axes[0].set_title('Content'); axes[0].axis('off')
for ax, (s, (img, t)) in zip(axes[1:], strength_results.items()):
    ax.imshow(img); ax.set_title(f'SD img2img\nstrength={s}  {t:.1f}s'); ax.axis('off')
plt.suptitle('SD img2img stylization — Van Gogh prompt', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'img2img_styles.png'), dpi=80)
plt.show()

In [ ]:
# Section 2.5 (cont.) — Final three-way comparison grid
# Fixed picks, not a quality ranking: mosaic is the fast-NST style that reads
# best at thumbnail size, and strength=0.65 is the img2img setting that keeps
# the apple recognisable while clearly restyling it. Swap either one to compare
# a different pair.

fast_name = 'mosaic'
best_fast, best_fast_t = fast_results[fast_name]
best_img2img = strength_results[0.65][0]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, img) in zip(axes, [
    ('Content',                      content_pil.resize((512, 512))),
    ('Classic NST\n(VGG19, 300 steps)', result_classic),
    (f'Fast NST\n({fast_name}, ~{best_fast_t*1000:.0f} ms)', best_fast),
    ('SD img2img\n(strength=0.65)',   best_img2img),
]):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle('Three-way comparison: classic NST vs. fast NST vs. SD img2img', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threeway_comparison.png'), dpi=100)
plt.show()

**What do you observe?**  
- Which method most faithfully reproduced the style *image* (Starry Night textures)?
- Which method preserved the content image's structure most accurately?
- Which result would you prefer for a real application, and does your answer change depending on the use case?

*Write your observation here:*

(double-click to edit)

---
## Part 3 — Exercises

### Exercise 1 — Swap content and style

**Task:** Re-run classic NST but swap the roles: use the style image (Starry Night) as the content and your original photo as the style reference. Does it work? Why might the results look different from the original direction? Show results at 300 steps with β=1e6.

**Expected output:** convergence grid and a one-paragraph analysis.

In [ ]:
# Exercise 1 -- Swap content/style roles
# Reload VGG first (we deleted it for VRAM)
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise1.png"))
# ...

*Does swapping work? How does the result differ from the original direction, and why?*

(double-click to edit)

### Exercise 2 — Content layer ablation

**Task:** Repeat classic NST with the same β=1e6 but change the content layer from `conv4_2` (index 21) to `conv2_2` (index 9) — an earlier, more spatially detailed layer. Show the results side-by-side. How does the output change when the model is asked to preserve finer detail rather than high-level structure?

**Expected output:** side-by-side of `conv4_2` vs. `conv2_2` results with a written analysis.

In [ ]:
# Exercise 2 -- Content layer ablation
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise2.png"))
# ...

*How does an earlier content layer change the style transfer result?*

(double-click to edit)

---
## Part 4 — Critical Analysis

> Required. Answer with real outputs from today's session.

**4.1 — Gram matrix intuition:** in your own words, explain why the Gram matrix captures *style* but not *spatial layout*. Use a specific example from your Section 2.2 result: identify a texture or color from the style image that appeared in the output image in a different spatial location than it was in the original style image.

*Write here:*


---

**4.2 — Style weight sweet spot:** from your Section 2.3 sweep, report the β value you found optimal for your content/style pair, and describe the specific visual quality that disappears first when β is too high. Why does high β destroy content? (Hint: consider what the gradient update does to the pixel values when style loss dominates.)

*Write here (include actual β value and image evidence):*


---

**4.3 — Method selection:** based on your three-way comparison, rank the three methods (classic NST, fast NST, SD img2img) by: (a) fidelity to the style reference image, (b) preservation of the content structure, (c) output speed. Name a deployment scenario where each method is the clear winner.

*Write here (reference threeway_comparison.png):*


---

**4.4 — VGG re-use:** this lesson used the exact same VGG19 weights that M4 used for transfer learning on image classification. What does this tell you about the generality of features learned from ImageNet classification? If you replaced VGG19 with a Vision Transformer (ViT), would the Gram matrix approach still work? Explain why or why not.

*Write here:*


---
## Submission Checklist

- [ ] All cells ran from start to finish without errors
- [ ] Section 2.2: classic NST convergence grid saved
- [ ] Section 2.3: style weight sweep (4 values) saved
- [ ] Section 2.4: fast NST grid (4 styles) saved
- [ ] Section 2.5: three-way comparison grid saved
- [ ] Exercise 1: swapped content/style with analysis
- [ ] Exercise 2: content layer ablation with analysis
- [ ] Part 4: Critical Analysis completed (all 4 questions)
- [ ] Cleanup cell run (see below)

**Save:** `File > Save a copy in Drive`

---
## Drive Cleanup — Run after submitting

VGG19 weights (~550 MB) are cached in Drive by torchvision. They are not needed after L08. Run this cell to free space before L09.

In [ ]:
# Drive cleanup — run after saving and submitting the notebook
import shutil, os, subprocess

# VGG19 weights stored by torchvision in TORCH_HOME/hub/checkpoints/
to_delete = [
    os.path.join(os.environ['TORCH_HOME'], 'hub', 'checkpoints', 'vgg19-dcbb9e9d.pth'),  # ~550 MB
]

for path in to_delete:
    if os.path.exists(path):
        os.remove(path)
        print(f'Deleted: {path}')
    else:
        print(f'Not found (may already be deleted): {path}')

result = subprocess.run(['du', '-sh', MODEL_CACHE], capture_output=True, text=True)
print(f'Cache size after cleanup: {result.stdout.strip()}')

---
## Before You Close This Tab

- [ ] Confirmed all outputs from this session are saved in `TAE_IA_M6/L08_output/` on Drive (see checklist above)
- [ ] Ran the Drive cleanup cell above (VGG19 only needed for this lesson)
- [ ] Disconnected and deleted this runtime: `Runtime > Disconnect and delete runtime`

Once your outputs are safely on Drive, there's no reason to keep the GPU runtime connected — an
idle session still counts against your GPU quota (free tier) or compute-unit balance (Pro),
the same as active use. Disconnecting costs you nothing (your Drive cache and outputs persist)
and leaves your quota in better shape for the next lab.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L08*  
*Platform: Google Colab (T4 GPU) · Python 3.10*